# 04. Phase fusion

B00 한 영상의 저장된 phase, Pose/angle, plate 결과를 다시 결합하고 검토용 시각화를 만듭니다. 모델을 다시 실행하지 않으며, 결과는 자세 점수나 일반화 성능이 아닙니다.

In [ ]:
from pathlib import Path
import json

import pandas as pd

from snatch_analysis.pipeline import build_frame_basis, fuse_frame_analysis, save_frame_analysis, save_json_atomically
from snatch_analysis.video import read_video_properties
from snatch_analysis.visualization import (
    save_angle_plot,
    save_phase_storyboard,
    save_plate_trajectory_plot,
    write_overlay,
)

VIDEO_ID = 'b00_sn_f_01'
RERUN_MODELS = False
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / VIDEO_ID
VIDEO_PATH = PROJECT_ROOT / 'data' / 'raw' / 'videos' / f'{VIDEO_ID}.mp4'
PHASE_PATH = ARTIFACT_DIR / 'phase_by_frame.csv'
PHASE_EVENTS_PATH = ARTIFACT_DIR / 'phase_events.csv'
POSE_PATH = ARTIFACT_DIR / 'pose_keypoints.csv'
ANGLE_PATH = ARTIFACT_DIR / 'joint_angles.csv'
PLATE_PATH = ARTIFACT_DIR / 'plate_track.csv'
SUMMARY_PATH = ARTIFACT_DIR / 'run_summary.json'
FRAME_ANALYSIS_PATH = ARTIFACT_DIR / 'frame_analysis.csv'


## 1. 필요한 저장 결과 확인

누락된 파일은 어떤 이전 노트북을 실행해야 하는지 함께 표시합니다.

In [ ]:
if RERUN_MODELS:
    raise RuntimeError('이 노트북은 cached CSV만 사용합니다. RERUN_MODELS=False로 두세요.')

required_artifacts = {
    PHASE_PATH: '01_data_diagnosis.ipynb와 phase annotation 단계',
    PHASE_EVENTS_PATH: 'phase annotation 단계',
    POSE_PATH: '02_pose_baseline.ipynb',
    ANGLE_PATH: '02_pose_baseline.ipynb',
    PLATE_PATH: '03_plate_detection.ipynb',
    SUMMARY_PATH: '02_pose_baseline.ipynb 또는 03_plate_detection.ipynb',
    VIDEO_PATH: '원본 B00 동영상',
}
missing_artifacts = [f'{path.name}: {next_step}' for path, next_step in required_artifacts.items() if not path.exists()]
if missing_artifacts:
    raise FileNotFoundError('필수 artifact가 없습니다. 먼저 실행: ' + '; '.join(missing_artifacts))
print('required artifacts: OK')


## 2. cached CSV/JSON 불러오기

이 단계는 Ultralytics 모델이나 가중치를 불러오지 않습니다.

In [ ]:
phase_by_frame = pd.read_csv(PHASE_PATH)
phase_events = pd.read_csv(PHASE_EVENTS_PATH)
pose_keypoints = pd.read_csv(POSE_PATH)
joint_angles = pd.read_csv(ANGLE_PATH)
plate_track = pd.read_csv(PLATE_PATH)
run_summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
video_properties = read_video_properties(VIDEO_PATH)
if video_properties['read_status'] != 'ok':
    raise RuntimeError(f'원본 영상을 읽을 수 없습니다: {VIDEO_PATH.name}')


## 3. 저장된 이벤트와 계산된 구간

아래 frame은 phase boundary입니다. storyboard의 representative frame과 혼동하지 않습니다.

In [ ]:
expected_event_frames = {
    'bar_off_floor': 0,
    'knee_pass': 65,
    'max_extension': 83,
    'catch_stable': 155,
    'recovery_stable': 225,
}
if phase_events['event_name'].duplicated().any():
    raise RuntimeError('B00 phase_events에 중복 event_name이 있습니다.')
observed_event_frames = dict(zip(phase_events['event_name'], phase_events['frame_index'], strict=True))
if observed_event_frames != expected_event_frames:
    raise RuntimeError('B00 저장 phase event 경계가 예상과 다릅니다.')
phase_ranges = phase_by_frame.groupby('phase', sort=False)['frame_index'].agg(start_frame='min', end_frame='max', frame_count='size')
expected_phase_ranges = {
    'First Pull': (0, 64),
    'Second Pull': (65, 82),
    'Drop Under & Catch': (83, 155),
    'Recovery': (156, 253),
}
observed_phase_ranges = {phase: (int(row.start_frame), int(row.end_frame)) for phase, row in phase_ranges.iterrows()}
recovery_start = int(phase_by_frame.loc[phase_by_frame['recovery_complete'], 'frame_index'].min())
if observed_phase_ranges != expected_phase_ranges or recovery_start != 225:
    raise RuntimeError('B00 저장 phase 범위 또는 recovery_complete 경계가 예상과 다릅니다.')
display(phase_events)
display(phase_ranges)
print(f'recovery_complete starts at frame {recovery_start}')


## 4. key/row 검증과 254행 fusion

Pose long-form은 요약한 뒤에만 frame basis에 left join합니다. 좌표와 각도, 원판의 결측값은 보간하지 않습니다.

In [ ]:
frame_count = int(video_properties['frame_count'])
fps = float(video_properties['fps'])
frame_basis = build_frame_basis(VIDEO_ID, frame_count, fps)
frame_analysis = fuse_frame_analysis(
    frame_basis, phase_by_frame, pose_keypoints, joint_angles, plate_track
)
if len(frame_analysis) != 254 or frame_analysis['frame_index'].tolist() != list(range(254)):
    raise RuntimeError('B00 fusion은 frame_index 0..253의 254행이어야 합니다.')
frame_analysis[['video_id', 'frame_index', 'phase', 'pose_status', 'plate_status']].head()


## 5. frame_analysis.csv 저장과 재로드

측정 CSV는 시각화보다 먼저 원자적으로 저장되므로 overlay 실패가 측정 성공을 지우지 않습니다.

In [ ]:
save_frame_analysis(frame_analysis, FRAME_ANALYSIS_PATH)
reloaded_frame_analysis = pd.read_csv(FRAME_ANALYSIS_PATH)
if len(reloaded_frame_analysis) != len(frame_analysis):
    raise RuntimeError('frame_analysis 재로드 행 수가 다릅니다.')


## 6. 저장값으로 plot, storyboard, overlay 재생성

각 결과는 독립적으로 상태를 기록합니다. 누락된 원판 점은 궤적에서 빈 구간으로 남습니다.

In [ ]:
visualization_jobs = {
    'angle_plot': lambda: save_angle_plot(FRAME_ANALYSIS_PATH, ARTIFACT_DIR / 'angle_plot.png'),
    'bar_path_plot': lambda: save_plate_trajectory_plot(FRAME_ANALYSIS_PATH, ARTIFACT_DIR / 'bar_path_plot.png'),
    'phase_storyboard': lambda: save_phase_storyboard(VIDEO_PATH, FRAME_ANALYSIS_PATH, ARTIFACT_DIR / 'phase_storyboard.png'),
    'overlay': lambda: write_overlay(VIDEO_PATH, FRAME_ANALYSIS_PATH, POSE_PATH, ARTIFACT_DIR / 'overlay.mp4'),
}
visualization_status = {}
for name, create_output in visualization_jobs.items():
    try:
        create_output()
    except Exception as error:
        visualization_status[name] = {'status': 'failed', 'reason': str(error)}
    else:
        visualization_status[name] = {'status': 'ok', 'reason': ''}
visualization_status


## 7. 결측과 시각화 상태 요약

낮은 신뢰도·탐지 누락은 N/A 또는 해당 status/reason으로 남기고, 다른 측정값을 채우지 않습니다.

In [ ]:
pose_na_count = int((reloaded_frame_analysis['pose_status'] == 'not_available').sum())
pose_partial_count = int((reloaded_frame_analysis['pose_status'] == 'partial').sum())
angle_status_columns = [column for column in reloaded_frame_analysis.columns if column.endswith('_status') and column not in {'pose_status', 'plate_status'}]
angle_na_count = int((reloaded_frame_analysis[angle_status_columns] != 'ok').any(axis=1).sum())
plate_missing = reloaded_frame_analysis['plate_status'] != 'ok'
longest_missing_gap = 0
for _, run in plate_missing.groupby(plate_missing.ne(plate_missing.shift()).cumsum()):
    if bool(run.iloc[0]):
        longest_missing_gap = max(longest_missing_gap, len(run))
confidence_columns = [column for column in reloaded_frame_analysis.columns if column.endswith('_confidence')]
worst_confidence = pd.to_numeric(reloaded_frame_analysis[confidence_columns].stack(), errors='coerce').min()
summary = {
    'pose_na_frames': pose_na_count,
    'pose_partial_frames': pose_partial_count,
    'angle_na_frames': angle_na_count,
    'plate_missing_or_confirmation_required_frames': int(plate_missing.sum()),
    'worst_confidence': None if pd.isna(worst_confidence) else float(worst_confidence),
    'longest_plate_missing_gap_frames': longest_missing_gap,
    'visualization_status': visualization_status,
}
run_summary['fusion'] = {'status': 'ok', 'frame_count': len(reloaded_frame_analysis), **summary}
save_json_atomically(run_summary, SUMMARY_PATH)
pd.Series(summary)


## 8. 해석 범위

이 결과는 B00 단일 영상의 pipeline smoke evidence입니다. Pose와 plate가 같은 영상에서 생성·재생된 사실은 일반화 성능이 아니며, 바벨 궤적과 관절 각도는 자세 점수·합격/불합격 피드백으로 사용하지 않습니다.